## 1. 라이브러리 및 환경 설정

In [10]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 30,
    'LEARNING_RATE': 1e-3,
    'BATCH_SIZE': 32,
    'SEED': 42
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. 데이터 로드 및 학습/검증 데이터 분할

In [11]:
# 1. 데이터 로드
train_df = pd.read_csv('./data/train.csv')
val_df = pd.read_csv('./data/dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


## 3. 커스텀 데이터셋 클래스 정의 및 데이터 로더

In [12]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        # 폴더 경로 설정
        folder_path = os.path.join(self.root_dir, sample_id)
        
        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)
            
        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views
        
        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [13]:
train_transform = transforms.Compose([
    ## Your Agumentation
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 학습/검증 세트 준비 (is_test=False 설정)
train_dataset = MultiViewDataset(train_df, './data/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, './data/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 2. 테스트 세트 준비 (is_test=True 설정)
test_df = pd.read_csv('./data/sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './data/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

## 4. 모델 정의 (Improved Multi-View with Cross-Attention)

In [14]:
# ─── Building Blocks ─────────────────────────────────────────────────

class CrossViewAttention(nn.Module):
    """Bidirectional cross-attention between front and top view features."""
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.cross_attn_f2t = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_t2f = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm_f = nn.LayerNorm(embed_dim)
        self.norm_t = nn.LayerNorm(embed_dim)
        self.ffn_f = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.ffn_t = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.norm_ffn_f = nn.LayerNorm(embed_dim)
        self.norm_ffn_t = nn.LayerNorm(embed_dim)

    def forward(self, feat_front, feat_top):
        attn_f, _ = self.cross_attn_f2t(feat_front, feat_top, feat_top)
        feat_front = self.norm_f(feat_front + attn_f)
        attn_t, _ = self.cross_attn_t2f(feat_top, feat_front, feat_front)
        feat_top = self.norm_t(feat_top + attn_t)
        feat_front = self.norm_ffn_f(feat_front + self.ffn_f(feat_front))
        feat_top = self.norm_ffn_t(feat_top + self.ffn_t(feat_top))
        return feat_front, feat_top


class MultiScaleExtractor(nn.Module):
    """Multi-scale feature extraction from ResNet-18 (layer2/3/4)."""
    def __init__(self, project_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2  # 128-d
        self.layer3 = resnet.layer3  # 256-d
        self.layer4 = resnet.layer4  # 512-d

        self.proj2 = nn.Conv2d(128, project_dim, 1)
        self.proj3 = nn.Conv2d(256, project_dim, 1)
        self.proj4 = nn.Conv2d(512, project_dim, 1)
        self.scale_weights = nn.Parameter(torch.ones(3) / 3)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        s2 = self.layer2(x)
        s3 = self.layer3(s2)
        s4 = self.layer4(s3)

        target_size = s4.shape[-2:]
        p2 = F.adaptive_avg_pool2d(self.proj2(s2), target_size)
        p3 = F.adaptive_avg_pool2d(self.proj3(s3), target_size)
        p4 = self.proj4(s4)

        w = F.softmax(self.scale_weights, dim=0)
        fused = w[0] * p2 + w[1] * p3 + w[2] * p4

        B, C, H, W = fused.shape
        spatial = fused.view(B, C, H * W).permute(0, 2, 1)  # [B, 49, D]
        global_feat = fused.mean(dim=[2, 3])                 # [B, D]
        return spatial, global_feat


class ViewDiscrepancyModule(nn.Module):
    """Encodes cross-view discrepancy as an explicit anomaly signal."""
    def __init__(self, feat_dim):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 4, feat_dim), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, feat_front, feat_top):
        diff = feat_front - feat_top
        hadamard = feat_front * feat_top
        combined = torch.cat([feat_front, feat_top, diff, hadamard], dim=1)
        return self.fusion(combined)


# ─── Main Model ──────────────────────────────────────────────────────

class CollapsePredictor(nn.Module):
    """
    Improved Multi-View Collapse Prediction Network.
    
    Improvements over baseline (simple concat):
    1. Multi-Scale Features: texture + part + semantic level
    2. Cross-View Attention: front ↔ top interaction
    3. View Discrepancy: explicit diff/product encoding
    4. View-Type Embedding: shared backbone + view identity
    
    Drop-in compatible: forward() returns tensor, same as baseline.
    """
    def __init__(self, num_classes=1, project_dim=256, num_attn_layers=2, num_heads=4):
        super().__init__()

        # Shared multi-scale backbone
        self.extractor = MultiScaleExtractor(project_dim=project_dim)

        # View-type embedding (lets shared backbone distinguish front vs top)
        self.view_embed = nn.Parameter(torch.randn(2, 1, project_dim) * 0.02)

        # Cross-view attention stack
        self.cross_attn_layers = nn.ModuleList([
            CrossViewAttention(project_dim, num_heads=num_heads)
            for _ in range(num_attn_layers)
        ])

        # Discrepancy module
        self.discrepancy = ViewDiscrepancyModule(project_dim)

        # Classifier: cross_f + cross_t + disc + global_f + global_t = 5D
        self.classifier = nn.Sequential(
            nn.Linear(project_dim * 5, project_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(project_dim, project_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(project_dim // 2, num_classes),
        )

    def forward(self, views):
        """
        Args:
            views: [front_tensor, top_tensor], each [B, 3, 224, 224]
        Returns:
            logits: [B, num_classes] — same interface as baseline
        """
        front_img, top_img = views[0], views[1]

        # Multi-scale feature extraction (shared backbone)
        spatial_f, global_f = self.extractor(front_img)
        spatial_t, global_t = self.extractor(top_img)

        # Add view-type embeddings
        spatial_f = spatial_f + self.view_embed[0]
        spatial_t = spatial_t + self.view_embed[1]

        # Cross-view attention
        sf, st = spatial_f, spatial_t
        for attn_layer in self.cross_attn_layers:
            sf, st = attn_layer(sf, st)

        cross_f = sf.mean(dim=1)  # [B, D]
        cross_t = st.mean(dim=1)  # [B, D]

        # View discrepancy
        disc = self.discrepancy(global_f, global_t)  # [B, D]

        # Classification
        combined = torch.cat([cross_f, cross_t, disc, global_f, global_t], dim=1)
        return self.classifier(combined)

## 4. 학습 및 검증 루프 구현

In [6]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()
        
        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()
            
            outputs = model(views).view(-1)
            # 1. 시그모이드를 통과시켜 확률값(unstable일 확률)으로 변환
            probs = torch.sigmoid(outputs)
            
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    
    # 대회 공식 LOGLOSS 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # Binary Log Loss 공식 직접 적용
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    
    # Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)
    
    return logloss_score, acc_score

In [15]:
model = CollapsePredictor().to(device)
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])

# --- Main Loop ---
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)
    
    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

Validation: 100%|██████████| 4/4 [00:00<00:00,  5.05it/s]


Epoch [1]
  - Train Loss: 0.3179
  - Val Log-Loss: 5.420757 | Val Acc: 0.6300


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.94it/s]


Epoch [2]
  - Train Loss: 0.0804
  - Val Log-Loss: 5.366968 | Val Acc: 0.5300


Validation: 100%|██████████| 4/4 [00:00<00:00,  5.08it/s]


...

Epoch [28]
  - Train Loss: 0.0132
  - Val Log-Loss: 5.088911 | Val Acc: 0.6700


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.89it/s]


Epoch [29]
  - Train Loss: 0.9180
  - Val Log-Loss: 17.856177 | Val Acc: 0.4800


Validation: 100%|██████████| 4/4 [00:00<00:00,  4.78it/s]

Epoch [30]
  - Train Loss: 0.1917
  - Val Log-Loss: 5.368791 | Val Acc: 0.5900


## 5. 추론 및 제출 파일 생성

In [16]:
model.eval()
all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]
        
        # 모델 출력 (Logit) -> 시그모이드 -> unstable(1)일 확률
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)
        
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
# 결과 저장 (컬럼  순서 중요)
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,  # unstable일 확률 저장
    'stable_prob': 1.0 - all_probs # stable일 확률 저장
})

os.makedirs("results", exist_ok=True)
submission.to_csv('./results/submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료.")

Inference: 100%|██████████| 32/32 [00:09<00:00,  3.42it/s]

submission.csv 저장 완료.
